In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path('/home/swl/braincell-ion_dyn').resolve()
if str(repo_root) in sys.path:
    sys.path.remove(str(repo_root))
sys.path.insert(0, str(repo_root))

os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import brainstate
import brainunit as u
import matplotlib.pyplot as plt
import numpy as np

import braincell
from braincell import Branch, Cell, Morphology
from braincell.filter import BranchSlice, at
from braincell.mech import CurrentClamp, Ion, MechanismProbe, CurrentProbe

print('braincell version:', braincell.__version__)
print('braincell import path:', Path(braincell.__file__).resolve())

brainstate.environ.set(precision=64)


In [ ]:
mod_dir = Path(braincell.__file__).resolve().parent.parent / 'examples' / 'neuron_compare' / 'Cerebellum_mod' / 'DCN'
ion_mod = mod_dir / 'ion' / 'ToyCaPumpFactorKinetic_SU15_DCN.mod'
channel_mod = mod_dir / 'channel' / 'CaHVA_SU15_DCN.mod'
print('mod_dir:', mod_dir)
print('ion mod exists:', ion_mod.exists())
print('channel mod exists:', channel_mod.exists())


In [ ]:
dt_ms = 0.01
duration_ms = 40.0
temperature_celsius = 36.0
v_init_mV = -60.0
kf_per_mM_ms = 2.0
kb_per_ms = 0.5
k_rel_per_ms = 0.05
pump_tot_mM_um = 1.0
kCa_per_coulomb = 3.45e-7
depth_um = 0.2
cyt_volume_um3 = 3.0
pump_area_um2 = 3.0
ci0_mM = 0.2
pumpbound0_mM_um = 0.3
pumpfree0_mM_um = pump_tot_mM_um - pumpbound0_mM_um
perm_cm_s = 7.5e-6
stim_delay_ms = 2.0
stim_dur_ms = 20.0
stim_amp_nA = 0.05

print({
    'dt_ms': dt_ms,
    'kf': kf_per_mM_ms,
    'kb': kb_per_ms,
    'k_rel': k_rel_per_ms,
    'PumpTot': pump_tot_mM_um,
    'kCa': kCa_per_coulomb,
    'depth_um': depth_um,
    'cyt_volume_um3': cyt_volume_um3,
    'pump_area_um2': pump_area_um2,
    'Ci0': ci0_mM,
    'PumpBound0': pumpbound0_mM_um,
})


In [ ]:
from neuron import h, load_mechanisms

if not load_mechanisms(str(mod_dir.resolve())):
    raise RuntimeError(f'NEURON mechanisms not found under {mod_dir!s}; compile the DCN mods first.')
h.load_file('stdrun.hoc')

sec = h.Section(name='soma')
sec.L = 20.0
sec.diam = 20.0
sec.nseg = 1
seg = sec(0.5)
sec.insert('ToyCaPumpFactorKinetic_SU15_DCN')
sec.insert('CaHVA_SU15_DCN')
ion_mech = seg.ToyCaPumpFactorKinetic_SU15_DCN
channel_mech = seg.CaHVA_SU15_DCN

ion_mech.kf = kf_per_mM_ms
ion_mech.kb = kb_per_ms
ion_mech.k_rel = k_rel_per_ms
ion_mech.PumpTot = pump_tot_mM_um
ion_mech.kCa = kCa_per_coulomb
ion_mech.depth = depth_um
ion_mech.cyt_volume = cyt_volume_um3
ion_mech.pump_area = pump_area_um2
channel_mech.perm = perm_cm_s

stim = h.IClamp(seg)
stim.delay = stim_delay_ms
stim.dur = stim_dur_ms
stim.amp = stim_amp_nA

h.celsius = temperature_celsius
h.dt = dt_ms
h.tstop = duration_ms
h.v_init = v_init_mV

t_vec = h.Vector().record(h._ref_t)
cai_vec = h.Vector().record(seg._ref_cai)
pumpbound_vec = h.Vector().record(ion_mech._ref_pumpbound)
pumpfree_vec = h.Vector().record(ion_mech._ref_pumpfree)
ica_vec = h.Vector().record(seg._ref_ica)
m_vec = h.Vector().record(channel_mech._ref_m)

h.finitialize(h.v_init)
seg.cai = ci0_mM
ion_mech.pumpbound = pumpbound0_mM_um
ion_mech.pumpfree = pumpfree0_mM_um
h.frecord_init()
h.continuerun(h.tstop)

neuron_t_ms = np.asarray(t_vec)
neuron_cai_mM = np.asarray(cai_vec)
neuron_pumpbound = np.asarray(pumpbound_vec)
neuron_pumpfree = np.asarray(pumpfree_vec)
neuron_ica_mA_cm2 = -np.asarray(ica_vec)
neuron_m = np.asarray(m_vec)
neuron_total = neuron_pumpbound + neuron_pumpfree

print('NEURON end cai/pumpbound/pumpfree:', float(neuron_cai_mM[-1]), float(neuron_pumpbound[-1]), float(neuron_pumpfree[-1]))
print('NEURON max conserve drift:', float(np.max(np.abs(neuron_total - pump_tot_mM_um))))


In [ ]:
soma = Branch.from_lengths(lengths=[20.0] * u.um, radii=[10.0, 10.0] * u.um, type='soma')
morpho = Morphology.from_root(soma, name='soma')
region = BranchSlice(branch_index=0, prox=0.0, dist=1.0)

cell = Cell(morpho, V_init=v_init_mV * u.mV, solver='staggered')
cell.paint(
    region,
    Ion(
        'ToyCaPumpFactorKinetic_SU2015_DCN',
        name='ca_toy_factor',
        temp=u.celsius2kelvin(temperature_celsius),
        kf=kf_per_mM_ms / (u.mM * u.ms),
        kb=kb_per_ms / u.ms,
        k_rel=k_rel_per_ms / u.ms,
        PumpTot=pump_tot_mM_um * u.mM * u.um,
        kCa=kCa_per_coulomb / u.coulomb,
        depth=depth_um * u.um,
        cyt_volume=cyt_volume_um3 * u.um ** 3,
        pump_area=pump_area_um2 * u.um ** 2,
        Ci_initializer=ci0_mM * u.mM,
        PumpBound_initializer=pumpbound0_mM_um * u.mM * u.um,
    ),
)
cell.paint(
    region,
    braincell.mech.Channel(
        'CaHVA_SU2015_DCN',
        ion_name='ca_toy_factor',
        perm=perm_cm_s * (u.cm / u.second),
    ),
)
cell.place(
    at('soma', 0.5),
    CurrentClamp.step(stim_amp_nA * u.nA, stim_dur_ms * u.ms, delay=stim_delay_ms * u.ms),
    MechanismProbe(mechanism='ca_toy_factor', field='Ci'),
    MechanismProbe(mechanism='ca_toy_factor', field='PumpBound'),
    MechanismProbe(mechanism='ca_toy_factor', field='PumpFree'),
    MechanismProbe(mechanism='CaHVA_SU2015_DCN', field='m'),
    CurrentProbe(ion='ca_toy_factor', mechanism='CaHVA_SU2015_DCN'),
)
with brainstate.environ.context(precision=64):
    cell.init_state()
    cell.reset_state()
    result = cell.run(dt=dt_ms * u.ms, duration=duration_ms * u.ms)

cell_cai_mM = np.asarray(result.traces['soma(0.5)_ca_toy_factor_Ci'].to_decimal(u.mM))
cell_pumpbound = np.asarray(result.traces['soma(0.5)_ca_toy_factor_PumpBound'].to_decimal(u.mM * u.um))
cell_pumpfree = np.asarray(result.traces['soma(0.5)_ca_toy_factor_PumpFree'].to_decimal(u.mM * u.um))
cell_ica_mA_cm2 = np.asarray(result.traces['soma(0.5)_CaHVA_SU2015_DCN_current'].to_decimal(u.mA / (u.cm ** 2)))
cell_m = np.asarray(result.traces['soma(0.5)_CaHVA_SU2015_DCN_m'])
cell_total = cell_pumpbound + cell_pumpfree

print('Cell end Ci/PumpBound/PumpFree:', float(cell_cai_mM[-1]), float(cell_pumpbound[-1]), float(cell_pumpfree[-1]))
print('Cell max conserve drift:', float(np.max(np.abs(cell_total - pump_tot_mM_um))))


In [ ]:
compare_t_ms = neuron_t_ms[1:]
neuron_cai = neuron_cai_mM[1:]
neuron_pumpbound_cmp = neuron_pumpbound[1:]
neuron_pumpfree_cmp = neuron_pumpfree[1:]
neuron_ica = neuron_ica_mA_cm2[1:]
neuron_m_cmp = neuron_m[1:]
neuron_total_cmp = neuron_total[1:]

def summarize_error(y, ref):
    y = np.asarray(y)
    ref = np.asarray(ref)
    n = min(len(y), len(ref))
    diff = y[:n] - ref[:n]
    return {
        'mae': float(np.mean(np.abs(diff))),
        'rmse': float(np.sqrt(np.mean(diff ** 2))),
        'max_abs': float(np.max(np.abs(diff))),
    }

err_cai = summarize_error(cell_cai_mM, neuron_cai[:len(cell_cai_mM)])
err_pumpbound = summarize_error(cell_pumpbound, neuron_pumpbound_cmp[:len(cell_pumpbound)])
err_pumpfree = summarize_error(cell_pumpfree, neuron_pumpfree_cmp[:len(cell_pumpfree)])
err_ica = summarize_error(cell_ica_mA_cm2, neuron_ica[:len(cell_ica_mA_cm2)])
err_m = summarize_error(cell_m, neuron_m_cmp[:len(cell_m)])

fig, axes = plt.subplots(5, 1, figsize=(8, 14), sharex=True)

axes[0].plot(compare_t_ms[:len(neuron_cai)], neuron_cai, label='NEURON cai')
axes[0].plot(compare_t_ms[:len(cell_cai_mM)], cell_cai_mM, '--', label='BrainCell Ci')
axes[0].set_ylabel('cai / Ci (mM)')
axes[0].legend()

axes[1].plot(compare_t_ms[:len(neuron_pumpbound_cmp)], neuron_pumpbound_cmp, label='NEURON pumpbound')
axes[1].plot(compare_t_ms[:len(cell_pumpbound)], cell_pumpbound, '--', label='BrainCell PumpBound')
axes[1].set_ylabel('pumpbound')
axes[1].legend()

axes[2].plot(compare_t_ms[:len(neuron_pumpfree_cmp)], neuron_pumpfree_cmp, label='NEURON pumpfree')
axes[2].plot(compare_t_ms[:len(cell_pumpfree)], cell_pumpfree, '--', label='BrainCell PumpFree')
axes[2].plot(compare_t_ms[:len(neuron_total_cmp)], neuron_total_cmp, ':', label='NEURON total')
axes[2].plot(compare_t_ms[:len(cell_total)], cell_total, '-.', label='BrainCell total')
axes[2].set_ylabel('pump pool')
axes[2].legend()

axes[3].plot(compare_t_ms[:len(neuron_ica)], neuron_ica, label='-NEURON ica')
axes[3].plot(compare_t_ms[:len(cell_ica_mA_cm2)], cell_ica_mA_cm2, '--', label='BrainCell current')
axes[3].set_ylabel('ica (mA/cm^2)')
axes[3].legend()

axes[4].plot(compare_t_ms[:len(neuron_m_cmp)], neuron_m_cmp, label='NEURON m')
axes[4].plot(compare_t_ms[:len(cell_m)], cell_m, '--', label='BrainCell m')
axes[4].set_xlabel('time (ms)')
axes[4].set_ylabel('gate m')
axes[4].legend()

plt.suptitle('ToyCaPumpFactor + CaHVA SU2015 DCN: NEURON vs Cell.run')
plt.show()

print('cai/Ci error:', err_cai)
print('pumpbound error:', err_pumpbound)
print('pumpfree error:', err_pumpfree)
print('ica error:', err_ica)
print('m gate error:', err_m)
